# 📰 Fake News Detector — Exploratory Data Analysis & Model Experiments

**Author:** Fake News Detector Project  
**Dataset:** [Kaggle — Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)  
**Goal:** Build an end-to-end NLP pipeline to classify news articles as FAKE (0) or REAL (1)

---

## Table of Contents
1. Introduction & Problem Statement
2. Data Loading and Overview
3. Exploratory Data Analysis (EDA)
4. Text Preprocessing
5. Feature Engineering — TF-IDF
6. Model Training & Cross-Validation
7. Model Evaluation
8. Logistic Regression Deep Dive
9. Conclusions
10. Future Work

---
## 1. Introduction & Problem Statement

Misinformation spreads **6× faster** than truthful information on social media (Vosoughi et al., MIT 2018).  
Automated fake news detection is a critical NLP problem that combines:
- **Text classification** — supervised binary classification
- **Feature engineering** — TF-IDF, n-grams, stylometric signals
- **Model interpretability** — understanding which words drive predictions

We use the Kaggle Fake and Real News Dataset containing ~44,000 articles labelled as FAKE or REAL.

In [ ]:
# Standard library
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve,
)

# Path setup
PROJECT_ROOT = pathlib.Path('.').resolve().parent
DATA_DIR = PROJECT_ROOT / 'data'
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# Plot settings
plt.rcParams.update({
    'figure.dpi': 100,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('Set2')

PALETTE = {'FAKE': '#E63946', 'REAL': '#2DC653'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Environment ready ✓')

---
## 2. Data Loading and Overview

In [ ]:
# Load raw CSVs
fake_df = pd.read_csv(DATA_DIR / 'Fake.csv')
real_df = pd.read_csv(DATA_DIR / 'True.csv')

print('FAKE dataset:')
print(f'  Shape: {fake_df.shape}')
print(f'  Columns: {list(fake_df.columns)}')
print(f'  Null values:\n{fake_df.isnull().sum()}\n')

print('REAL dataset:')
print(f'  Shape: {real_df.shape}')
print(f'  Columns: {list(real_df.columns)}')
print(f'  Null values:\n{real_df.isnull().sum()}')

In [ ]:
print('Sample FAKE articles:')
fake_df.head(3)

In [ ]:
print('Sample REAL articles:')
real_df.head(3)

In [ ]:
# Add labels and merge
fake_df['label'] = 0  # FAKE
real_df['label'] = 1  # REAL

df = pd.concat([fake_df, real_df], ignore_index=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# Combine title + text
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df.dropna(subset=['content', 'label'], inplace=True)

print(f'Total articles: {len(df):,}')
print(f'FAKE articles:  {(df.label==0).sum():,} ({(df.label==0).mean()*100:.1f}%)')
print(f'REAL articles:  {(df.label==1).sum():,} ({(df.label==1).mean()*100:.1f}%)')
df.dtypes

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count chart
counts = df['label'].value_counts().rename({0: 'FAKE', 1: 'REAL'})
axes[0].bar(counts.index, counts.values, color=[PALETTE['FAKE'], PALETTE['REAL']], alpha=0.85)
axes[0].set_title('Class Distribution (Counts)', fontsize=13)
axes[0].set_ylabel('Number of Articles')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts.values,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=[PALETTE['FAKE'], PALETTE['REAL']],
    startangle=90,
    pctdistance=0.85,
)
axes[1].set_title('Class Distribution (%)', fontsize=13)

plt.suptitle('Dataset Balance: FAKE vs REAL News', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Article word count distribution (FAKE vs REAL)
df['word_count'] = df['content'].apply(lambda t: len(str(t).split()))

fig, ax = plt.subplots(figsize=(11, 5))
for label, name, color in [(0, 'FAKE', PALETTE['FAKE']), (1, 'REAL', PALETTE['REAL'])]:
    subset = df[df['label'] == label]['word_count']
    ax.hist(subset, bins=60, alpha=0.55, color=color,
            label=f'{name}  (median={int(subset.median())} words)')

ax.set_xlabel('Word Count', fontsize=12)
ax.set_ylabel('Number of Articles', fontsize=12)
ax.set_title('Article Length Distribution — FAKE vs REAL', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(0)
plt.tight_layout()
plt.show()

print('Word Count Stats:')
print(df.groupby('label')['word_count'].describe().round(1))

In [ ]:
# 3.3 Subject / category distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, name, color in zip(axes, [0, 1], ['FAKE', 'REAL'], [PALETTE['FAKE'], PALETTE['REAL']]):
    counts = df[df['label'] == label]['subject'].value_counts()
    ax.barh(counts.index, counts.values, color=color, alpha=0.8)
    ax.set_title(f'{name} News — Subject Distribution', fontsize=12)
    ax.set_xlabel('Article Count')
    ax.invert_yaxis()

plt.suptitle('Subject/Category Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 WordCloud — FAKE News
fake_text = ' '.join(df[df['label'] == 0]['content'].astype(str))

wc = WordCloud(
    width=900, height=450,
    background_color='#1a1a2e',
    colormap='Reds',
    max_words=150,
    random_state=RANDOM_STATE,
).generate(fake_text)

plt.figure(figsize=(11, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud — FAKE News', fontsize=15, pad=15, color='#E63946', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.5 WordCloud — REAL News
real_text = ' '.join(df[df['label'] == 1]['content'].astype(str))

wc = WordCloud(
    width=900, height=450,
    background_color='#0d1f0f',
    colormap='Greens',
    max_words=150,
    random_state=RANDOM_STATE,
).generate(real_text)

plt.figure(figsize=(11, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud — REAL News', fontsize=15, pad=15, color='#2DC653', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Text Preprocessing

In [ ]:
from preprocess import clean_text, preprocess_dataframe

# Show step-by-step cleaning on a single example
raw = df['content'].iloc[0]
print('=== RAW TEXT (first 400 chars) ===')
print(raw[:400])

print('\n=== CLEANED TEXT ===')
print(clean_text(raw)[:400])

In [ ]:
# Apply full preprocessing (this may take ~2 minutes)
df = preprocess_dataframe(df)
print(f'Ready: {len(df):,} preprocessed articles')
df[['label', 'clean_content']].head(2)

---
## 5. Feature Engineering — TF-IDF

In [ ]:
# Train/test split
X = df['clean_content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

# Fit TF-IDF
vectorizer = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Feature matrix shape: {X_train_tfidf.shape}')
print(f'Sparsity: {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]))*100:.2f}%')

In [ ]:
# Show top TF-IDF features by variance across all documents
import scipy.sparse as sp
feature_names = np.array(vectorizer.get_feature_names_out())
mean_tfidf    = np.array(X_train_tfidf.mean(axis=0)).flatten()
top20_idx     = np.argsort(mean_tfidf)[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(20), mean_tfidf[top20_idx], color='#457B9D', alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels(feature_names[top20_idx])
ax.invert_yaxis()
ax.set_xlabel('Mean TF-IDF Weight')
ax.set_title('Top 20 Features by Mean TF-IDF Weight', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Model Training & Cross-Validation

In [ ]:
from sklearn.pipeline import Pipeline

models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_STATE),
    'Multinomial NB':      MultinomialNB(alpha=0.1),
    'Linear SVM':          SGDClassifier(loss='hinge', max_iter=1000, random_state=RANDOM_STATE, tol=1e-3),
    'Passive Aggressive':  PassiveAggressiveClassifier(max_iter=1000, random_state=RANDOM_STATE, tol=1e-3),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}  CV F1 = {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# CV scores visualized
fig, ax = plt.subplots(figsize=(10, 5))
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds  = [cv_results[n].std()  for n in names]
colors = ['#457B9D', '#E63946', '#2DC653', '#F4A261']

bars = ax.bar(names, means, color=colors, alpha=0.85, yerr=stds, capsize=6)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.001,
            f'{m:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0.9, 1.02)
ax.set_ylabel('F1 Score (5-Fold CV)')
ax.set_title('5-Fold Cross-Validation F1 Scores ± Std Dev', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Model Evaluation on Held-Out Test Set

In [ ]:
# Train all models on full training set and evaluate on test set
fitted = {}
results = []

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    fitted[name] = model
    
    y_pred = model.predict(X_test_tfidf)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_tfidf)[:, 1]
    else:
        s = model.decision_function(X_test_tfidf)
        y_proba = (s - s.min()) / (s.max() - s.min() + 1e-9)
    
    results.append({
        'model': name,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'y_pred':    y_pred,
        'y_proba':   y_proba,
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('y_pred', 'y_proba')} for r in results])
results_df.set_index('model', inplace=True)
results_df.round(4)

In [ ]:
# Confusion matrix for Logistic Regression
lr_res = next(r for r in results if r['model'] == 'Logistic Regression')
cm = confusion_matrix(y_test, lr_res['y_pred'])

fig, ax = plt.subplots(figsize=(6, 5))
total = cm.sum()
annot = [[f'{v}\n({v/total*100:.1f}%)' for v in row] for row in cm]
sns.heatmap(cm, annot=annot, fmt='', cmap='RdYlGn',
            xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#E63946', '#457B9D', '#2DC653', '#F4A261']

for res, color in zip(results, colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, label=f"{res['model']} (AUC={res['roc_auc']:.4f})", color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — All Models', fontsize=13)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Full classification report for Logistic Regression
print('Classification Report — Logistic Regression')
print(classification_report(y_test, lr_res['y_pred'], target_names=['FAKE', 'REAL']))

In [ ]:
# Error analysis — 10 articles the model got wrong
lr_model = fitted['Logistic Regression']
y_pred_all = lr_model.predict(X_test_tfidf)
wrong_idx = np.where(y_pred_all != y_test.values)[0]

print(f'Total misclassified: {len(wrong_idx)} / {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)')
print('\n=== 10 Misclassified Examples ===')

X_test_list = X_test.values
for i in wrong_idx[:10]:
    true_label = 'REAL' if y_test.values[i] == 1 else 'FAKE'
    pred_label = 'REAL' if y_pred_all[i] == 1 else 'FAKE'
    print(f'True: {true_label} | Predicted: {pred_label}')
    print(f'Text snippet: {X_test_list[i][:150]}...')
    print()

---
## 8. Logistic Regression Deep Dive

In [ ]:
# Extract LR coefficients
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = lr_model.coef_[0]

# Most positive coefficients → REAL indicators
top_real_idx  = np.argsort(coefs)[-25:][::-1]
# Most negative coefficients → FAKE indicators
top_fake_idx  = np.argsort(coefs)[:25]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# REAL indicators
ax1.barh(range(25), coefs[top_real_idx], color='#2DC653', alpha=0.85)
ax1.set_yticks(range(25))
ax1.set_yticklabels(feature_names[top_real_idx], fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel('LR Coefficient')
ax1.set_title('Top 25 REAL News Indicators', fontsize=12)

# FAKE indicators
ax2.barh(range(25), np.abs(coefs[top_fake_idx]), color='#E63946', alpha=0.85)
ax2.set_yticks(range(25))
ax2.set_yticklabels(feature_names[top_fake_idx], fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('|LR Coefficient|')
ax2.set_title('Top 25 FAKE News Indicators', fontsize=12)

plt.suptitle('Logistic Regression Coefficient Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Conclusions

### Which model performed best?
- **Logistic Regression** and **Passive Aggressive** achieve near-perfect accuracy (~99%) on this dataset.
- Multinomial Naive Bayes, while simpler, lags slightly (~94%) because it assumes feature independence.

### Strongest fake news indicators
- Sensational language: *breaking*, *exposed*, *shocking*, *truth*
- Conspiracy framing: *deep state*, *mainstream media*, *big pharma*, *government suppressing*
- Call to action: *share before deleted*, *wake up*, *they don't want you to know*

### Limitations of TF-IDF approach
1. **No semantics**: TF-IDF treats words as independent tokens — it cannot understand *context* or *irony*
2. **Vocabulary shift**: Model trained on 2019 articles may not detect 2024 misinformation patterns
3. **Style only**: Detects stylometric patterns, not factual accuracy — a factually wrong article written in journalistic style might be misclassified
4. **English-only**: No cross-lingual capability
5. **No source credibility**: Same text published by NYT vs an anonymous blog would receive identical treatment

---
## 10. Future Work

| Enhancement | Expected Impact |
|---|---|
| **BERT / RoBERTa** | Contextual embeddings for richer semantic understanding |
| **Fact-checking API** | Cross-reference claims against Snopes, PolitiFact, ClaimBuster |
| **Source credibility** | Incorporate domain reputation scores (Media Bias Fact Check) |
| **Browser extension** | Real-time detection while browsing |
| **Multi-language** | Extend via multilingual BERT (mBERT) |
| **Ensemble** | Gradient Boosting + Transformer hybrid |
| **Explainability** | SHAP values for more robust feature attribution |